# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/keshav-geu/flyrank-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

### Baseline Rule

Pages should be prioritized for review if they have not been updated recently, still receive a high number of impressions, and have a relatively low CTR. These pages are likely to benefit from a content refresh or optimization.

### Reason Codes

- STALE_HIGH_TRAFFIC – Page is old but still receives many impressions.
- LOW_CTR – CTR is lower than expected for its visibility.
- REFRESH_CANDIDATE – Meets multiple conditions and should be reviewed first.

In [9]:
rule = "Prioritize pages that are old, receive high impressions, and have low CTR."

reason_codes = [
    "STALE_HIGH_TRAFFIC",
    "LOW_CTR",
    "REFRESH_CANDIDATE"
]

print("Rule:", rule)
print("Reason Codes:")
for code in reason_codes:
    print("-", code)

Rule: Prioritize pages that are old, receive high impressions, and have low CTR.
Reason Codes:
- STALE_HIGH_TRAFFIC
- LOW_CTR
- REFRESH_CANDIDATE


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [4]:
!git clone https://github.com/keshav-geu/flyrank-internship.git

Cloning into 'flyrank-internship'...
remote: Enumerating objects: 138, done.
remote: Counting objects: 100% (138/138), done.
remote: Compressing objects: 100% (94/94), done.
remote: Total 138 (delta 52), reused 91 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (138/138), 1.86 MiB | 10.50 MiB/s, done.
Resolving deltas: 100% (52/52), done.


In [5]:
%cd /content

!git clone https://github.com/keshav-geu/flyrank-internship.git

%cd flyrank-internship

!ls

/content
fatal: destination path 'flyrank-internship' already exists and is not an empty directory.
/content/flyrank-internship
AGENTS.md  DATA_USE.md	LICENSE    README.md	     SETUP.md	 work
CLAUDE.md  docs		notebooks  requirements.txt  skills
data	   GUIDE.md	outputs    scripts	     submission


In [6]:
import pandas as pd
import os

# Load data
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Simple baseline score
df["baseline_score"] = (
    df["impressions_90d"] / 1000
    + df["content_age_days"] / 180
    - df["ctr"] * 10
)

# Reason code
def reason(row):
    if row["content_age_days"] > 180 and row["impressions_90d"] > 1000:
        return "STALE_HIGH_TRAFFIC"
    elif row["ctr"] < 0.03:
        return "LOW_CTR"
    else:
        return "REFRESH_CANDIDATE"

df["reason_code"] = df.apply(reason, axis=1)

# Action
df["action"] = "Review"

# Rank
ranked = df.sort_values("baseline_score", ascending=False)

# Save CSV
os.makedirs("work/outputs", exist_ok=True)

ranked.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

ranked.head(20)

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,baseline_score,reason_code,action
6653,content_5fe46e04994d,client_4e07408562,1900.0,0.00,LOW,0.00,keyword article,informational,NaN,NaN,...,4.23,26.90,0.38,excellent,page_1,down,-44.8,519.298333,STALE_HIGH_TRAFFIC,Review
17812,content_aaef01a50def,client_19581e27de,4400.0,0.05,LOW,0.11,keyword article,informational,NaN,NaN,...,2.42,4.77,0.00,excellent,page_1,stable,-4.0,517.081222,STALE_HIGH_TRAFFIC,Review
26844,content_8c19996aa890,client_4e07408562,70.0,0.01,LOW,0.00,keyword article,informational,2895.0,19343.0,...,11.73,16.03,0.00,excellent,top_3,down,-44.5,510.224222,STALE_HIGH_TRAFFIC,Review
19636,content_2cb567c3c89b,client_6208ef0f77,0.0,0.00,LOW,0.00,keyword article,informational,6183.0,39587.0,...,5.82,5.64,1.21,excellent,page_3_5,up,23.1,497.577000,REFRESH_CANDIDATE,Review
21819,content_4c36c775b818,client_4e07408562,40.0,0.00,LOW,0.00,keyword article,informational,3097.0,20514.0,...,11.31,18.07,0.00,excellent,top_3,down,-33.2,461.475222,STALE_HIGH_TRAFFIC,Review
29400,content_2dba2b1f9536,client_6208ef0f77,0.0,0.00,LOW,0.00,keyword article,informational,7676.0,48015.0,...,2.73,7.77,0.43,excellent,page_3_5,stable,1.4,442.995111,STALE_HIGH_TRAFFIC,Review
29879,content_1a9e894be2e2,client_19581e27de,70.0,0.07,LOW,0.10,keyword article,transactional,NaN,NaN,...,2.37,4.97,0.00,excellent,page_1,down,-27.0,416.557778,STALE_HIGH_TRAFFIC,Review
18870,content_db5989a78dd3,client_4e07408562,110.0,0.00,LOW,0.00,keyword article,commercial,2682.0,17806.0,...,2.32,3.72,0.00,excellent,page_1,up,556.2,345.483222,STALE_HIGH_TRAFFIC,Review
13537,content_2c2606c5d176,client_19581e27de,590.0,0.18,LOW,0.31,keyword article,commercial,NaN,NaN,...,1.30,3.28,0.00,excellent,page_1,down,-36.5,344.110111,STALE_HIGH_TRAFFIC,Review
26531,content_cb112fce36be,client_19581e27de,70.0,0.65,MEDIUM,0.34,keyword article,transactional,2761.0,18472.0,...,2.08,2.95,0.00,excellent,page_1,down,-41.8,309.010000,REFRESH_CANDIDATE,Review


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [7]:
top20 = ranked.head(20)
top20

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,baseline_score,reason_code,action
6653,content_5fe46e04994d,client_4e07408562,1900.0,0.00,LOW,0.00,keyword article,informational,NaN,NaN,...,4.23,26.90,0.38,excellent,page_1,down,-44.8,519.298333,STALE_HIGH_TRAFFIC,Review
17812,content_aaef01a50def,client_19581e27de,4400.0,0.05,LOW,0.11,keyword article,informational,NaN,NaN,...,2.42,4.77,0.00,excellent,page_1,stable,-4.0,517.081222,STALE_HIGH_TRAFFIC,Review
26844,content_8c19996aa890,client_4e07408562,70.0,0.01,LOW,0.00,keyword article,informational,2895.0,19343.0,...,11.73,16.03,0.00,excellent,top_3,down,-44.5,510.224222,STALE_HIGH_TRAFFIC,Review
19636,content_2cb567c3c89b,client_6208ef0f77,0.0,0.00,LOW,0.00,keyword article,informational,6183.0,39587.0,...,5.82,5.64,1.21,excellent,page_3_5,up,23.1,497.577000,REFRESH_CANDIDATE,Review
21819,content_4c36c775b818,client_4e07408562,40.0,0.00,LOW,0.00,keyword article,informational,3097.0,20514.0,...,11.31,18.07,0.00,excellent,top_3,down,-33.2,461.475222,STALE_HIGH_TRAFFIC,Review
29400,content_2dba2b1f9536,client_6208ef0f77,0.0,0.00,LOW,0.00,keyword article,informational,7676.0,48015.0,...,2.73,7.77,0.43,excellent,page_3_5,stable,1.4,442.995111,STALE_HIGH_TRAFFIC,Review
29879,content_1a9e894be2e2,client_19581e27de,70.0,0.07,LOW,0.10,keyword article,transactional,NaN,NaN,...,2.37,4.97,0.00,excellent,page_1,down,-27.0,416.557778,STALE_HIGH_TRAFFIC,Review
18870,content_db5989a78dd3,client_4e07408562,110.0,0.00,LOW,0.00,keyword article,commercial,2682.0,17806.0,...,2.32,3.72,0.00,excellent,page_1,up,556.2,345.483222,STALE_HIGH_TRAFFIC,Review
13537,content_2c2606c5d176,client_19581e27de,590.0,0.18,LOW,0.31,keyword article,commercial,NaN,NaN,...,1.30,3.28,0.00,excellent,page_1,down,-36.5,344.110111,STALE_HIGH_TRAFFIC,Review
26531,content_cb112fce36be,client_19581e27de,70.0,0.65,MEDIUM,0.34,keyword article,transactional,2761.0,18472.0,...,2.08,2.95,0.00,excellent,page_1,down,-41.8,309.010000,REFRESH_CANDIDATE,Review


## 4. Weak picks + leakage check

Some lower-ranked pages may not actually require a refresh because high impressions alone do not guarantee outdated content. Seasonal traffic or recent manual updates could make these recommendations incorrect.

No future-window information, product flags, or label-derived columns were used in the baseline score.

In [8]:
ranked.tail(10)

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,baseline_score,reason_code,action
4606,content_3f3576c295f5,client_4ec9599fc2,0.0,0.0,LOW,0.0,keyword article,informational,NaN,NaN,...,0.0,0.0,0.0,low,top_3,flat,NaN,-997.921222,REFRESH_CANDIDATE,Review
18825,content_98458bafe297,client_d4735e3a26,NaN,NaN,NaN,NaN,feedly article,NaN,1316.0,9110.0,...,0.0,0.0,0.0,low,top_3,flat,NaN,-998.254556,REFRESH_CANDIDATE,Review
21147,content_6016b918a48f,client_d4735e3a26,NaN,NaN,NaN,NaN,feedly article,NaN,766.0,5837.0,...,0.0,0.0,0.0,low,page_3_5,flat,NaN,-998.265667,REFRESH_CANDIDATE,Review
19598,content_a84e013a5f94,client_d4735e3a26,NaN,NaN,NaN,NaN,feedly article,NaN,936.0,6558.0,...,0.0,0.0,0.0,low,page_1,flat,NaN,-998.337889,REFRESH_CANDIDATE,Review
7514,content_b1e4f7904d85,client_d4735e3a26,NaN,NaN,NaN,NaN,feedly article,NaN,791.0,5588.0,...,0.0,0.0,0.0,low,top_3,flat,NaN,-998.337889,REFRESH_CANDIDATE,Review
22607,content_bc2c0c7243df,client_d4735e3a26,NaN,NaN,NaN,NaN,feedly article,NaN,873.0,6239.0,...,0.0,0.0,0.0,low,top_3,flat,NaN,-998.365667,REFRESH_CANDIDATE,Review
13661,content_bf398aa7400e,client_d4735e3a26,NaN,NaN,NaN,NaN,feedly article,NaN,816.0,6204.0,...,0.0,0.0,0.0,low,top_3,flat,NaN,-998.365667,REFRESH_CANDIDATE,Review
19341,content_4272d3a330a3,client_9f14025af0,0.0,0.0,LOW,0.0,keyword article,informational,3005.0,20441.0,...,100.0,25.0,0.0,low,page_1,flat,NaN,-999.199000,REFRESH_CANDIDATE,Review
240,content_006b16e7a2e7,client_9f14025af0,0.0,0.0,LOW,0.0,keyword article,informational,2522.0,17424.0,...,100.0,50.0,0.0,low,top_3,flat,NaN,-999.221222,REFRESH_CANDIDATE,Review
6473,content_cfa4d9f1bf0a,client_d4735e3a26,NaN,NaN,NaN,NaN,feedly article,NaN,2972.0,21585.0,...,0.0,0.0,0.0,low,page_3_5,new,NaN,-999.376778,REFRESH_CANDIDATE,Review


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.